In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Natural Language Processing with Disaster Tweets

This notebook explores Disaster Tweets from Howard, devrishi, Culliton, and Guo (2019) to predict which tweets are about real disasters and which are not. This project aims to output 1 or 0 to indicate whether a disaster is real or not.

The goal of this project is to learn how to apply RNNs and NLP to real business scenarios, such as analyzing tweets.

# Exploratory Data Analysis (EDA)

Let us understand the data we have been given. Since this is tabular data, we can use the head function to check the first few rows and see the available columns (features).

In [ ]:
import pandas as pd

train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
train.head()

Next, we can check the shape of the data to get the size of the dataset we are dealing with, as well as some info about each of the columns.

In [ ]:
print(train.shape)
print(train.info())

We can see that there are 7613 rows and five features. The two features for this project that we will care about are text and target.

Next, we want to check for null values.

In [ ]:
print(train.isnull().sum())

Here, we can see that location is often missing from the tweets, and keywords are also missing sometimes. What we will do later is fill the nulls with missing values, even though we will not be using these later. It will be helpful in case we decide to add them.

In [ ]:
import matplotlib.pyplot as plt

train['target'].value_counts().plot(kind='bar', title='Target Distribution')
plt.xticks(ticks=[0, 1], labels=['Not Disaster', 'Disaster'], rotation=0)
plt.show()

From the plot above, the distribution of non-disaster and disaster tweets is not too imbalanced, which means our models should perform well.

In [ ]:
print(f"Unique keywords: {train['keyword'].nunique()}")
print(f"Unique locations: {train['location'].nunique()}")

# Most frequent keywords
train['keyword'].value_counts().head(10).plot(kind='barh', title='Top Keywords')
plt.show()

The number of unique keywords is not that big. Additionally, the number of unique locations is substantial, indicating a nearly one-to-two ratio of locations to tweets.

In [ ]:
train['text_length'] = train['text'].apply(len)
train['word_count'] = train['text'].apply(lambda x: len(x.split()))

train[['text_length', 'word_count']].hist(bins=30, figsize=(12, 5))
plt.suptitle("Text Length and Word Count Distribution")
plt.show()

Looking at the distribution above, it looks normal for the word count. The character count (text_length) is more clustered to the longer side, around 140.

# Preprocessing Data

What we will do now is clean the data. We will first remove URLs from the tweets, as we are not sure how relevant they will be to the machine. We will also remove elements such as mentions, hashtags, numbers, and punctuation and strip extra whitespace. Then, fill in the missing locations and keywords with placeholders.

In [ ]:
import re

def clean_text(text):
    text = text.lower()                             # Lowercase
    text = re.sub(r'http\S+|www\S+', '', text)      # Remove URLs
    text = re.sub(r'@\w+', '', text)                # Remove @mentions
    text = re.sub(r'#', '', text)                   # Remove hashtag symbols
    text = re.sub(r'\d+', '', text)                 # Remove numbers
    text = re.sub(r'[^a-z\s]', '', text)            # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()        # Normalize whitespace
    return text

# Remove urls from tweets
train['clean_text'] = train['text'].apply(clean_text)
test['clean_text'] = test['text'].apply(clean_text)

# Fill missing keywords/locations with a placeholder if needed
train['keyword'] = train['keyword'].fillna('no_keyword')
train['location'] = train['location'].fillna('no_location')

In [ ]:
print(train.isnull().sum())

We can now see that there are no null values. Let us create a quick table on how relevant keywords and location may be to the disaster.

In [ ]:
keyword_stats = train.groupby('keyword')['target'].agg(['count', 'mean']).sort_values(by='mean', ascending=False)
keyword_stats

Here, there could be some correlation.

In [ ]:
location_stats = train.groupby('location')['target'].agg(['count', 'mean']).sort_values(by='mean', ascending=False)
location_stats[location_stats['count'] > 20].sort_values(by='mean', ascending=False).head(10)

As well as for location. Let us see how that might look in a correlation matrix.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_keyword = LabelEncoder()
le_location = LabelEncoder()

train['keyword_encoded'] = le_keyword.fit_transform(train['keyword'])
train['location_encoded'] = le_location.fit_transform(train['location'])

import seaborn as sns
import matplotlib.pyplot as plt

correlation = train[['keyword_encoded', 'location_encoded', 'target']].corr()

sns.heatmap(correlation, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

We can see some relevance, but it is not very strong. So, we can safely ignore them for now.

## Preprocessing Tweets

To prepare the tweets for our models, we need to perform preprocessing first. In this case, we are using the Tokenizer from Keras. We have set a low vocabulary size to prevent a sparse dataset, and we have also set the maximum sequence length to 100.

After tokenization and padding, we will split the training dataset into a test and training set. Splitting the dataset will enable the model to validate its performance.

In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Step 1: Tokenization and Padding
max_vocab = 5000
max_len = 100

tokenizer = Tokenizer(num_words=max_vocab, oov_token="<OOV>")
tokenizer.fit_on_texts(train['clean_text'])

train_sequences = tokenizer.texts_to_sequences(train['clean_text'])

X = pad_sequences(train_sequences, maxlen=max_len, padding='post', truncating='post')

X_train, X_val, y_train, y_val = train_test_split(
    X, train['target'], test_size=0.2, random_state=42)

# Model Architecture

Let us create the model. We will use a very basic RNN. After that, we will compare it to LSTM and then proceed to GRU. After that, we will iterate over some hyperparameters.

## Simple RNN

In this architecture, we are using the SimpleRNN from Keras with no dropout layers. We are also using things like early stopping so that if the model does not improve, it will stop. We will reduce the learning rate when it plateaus. Finally, we will also save the best model so that we can restore it later for the final results. We will run for a maximum of 15 epochs.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

BATCH_SIZE = 64

simple_model = Sequential([
    Embedding(input_dim=max_vocab, output_dim=128),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])
simple_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

simple_model.summary()

## LSTM

LSTM, or Long Short-Term Memory, is supposed to capture long-term dependencies in the data. Since tweets are short, there is some uncertainty about whether this will improve over the simple RNN. However, let us proceed to compare it with the basic model.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

BATCH_SIZE = 64

lstm_model = Sequential([
    Embedding(input_dim=max_vocab, output_dim=100),
    LSTM(64, return_sequences=False),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

lstm_model.summary()

## GRU

Gated Recurrent Units (GRU) are also designed to target the vanishing gradient problem with RNNs. However, it does not handle the long-term dependencies as well as the LSTM.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

from tensorflow.keras.optimizers import Adam

BATCH_SIZE = 64

gru_model = Sequential([
    Embedding(input_dim=max_vocab, output_dim=128),
    GRU(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])

gru_model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0005),  # smaller LR for stability
    metrics=['accuracy']
)

gru_model.summary()

# Results and Analysis

Now, it is time to run our models and see if we can improve them. Firstly, we will run the model for the basic architecture. We will then use the confusion matrix and the Area Under the Curve (AUC) to check for the model's accuracy. We will also use the F1 score to evaluate, as that is what will be tested as part of the Kaggle competition.

## Simple Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

simple_callbacks_list = [
    # This helps stop the model early
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    # Reduces learning rate when plateued
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    # save best model
    ModelCheckpoint(
        filepath='basic_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

simple_history = simple_model.fit(
    X_train, 
    y_train, 
    epochs=15, 
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=simple_callbacks_list
)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(simple_history.history['accuracy'], label='Train Accuracy')
plt.plot(simple_history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(simple_history.history['loss'], label='Train Loss')
plt.plot(simple_history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

We can see that training accuracy increases, and validation (test) accuracy also increases. However, the loss for validation increased, while the loss for training decreased.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Predict on validation set
y_pred_probs = simple_model.predict(X_val)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# Step 2: Evaluation metrics
basic_acc = accuracy_score(y_val, y_pred)
basic_f1 = f1_score(y_val, y_pred)

print(f"Validation Accuracy: {basic_acc:.4f}")
print(f"Validation F1 Score: {basic_f1:.4f}")
print("\nClassification Report:\n", classification_report(y_val, y_pred))

# Step 3: Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
labels = ['Not Disaster', 'Disaster']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

We can see the results of this in the confusion matrix. It missed many disasters and also mispredicted non-disasters. 

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_val, y_pred)
basic_roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {basic_roc_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()

Examining the ROC and AUC above, it appears to be quite good. It is a start for our baseline, so we can only improve from here.

Let us compare it with the LSTM.

## LSTM Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

lstm_callbacks_list = [
    # This helps stop the model early
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    # Reduces learning rate when plateued
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    # save best model
    ModelCheckpoint(
        filepath='lstm_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

lstm_history = lstm_model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=lstm_callbacks_list
)


In [ ]:
import matplotlib.pyplot as plt

# Accuracy plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(lstm_history.history['accuracy'], label='Train Accuracy')
plt.plot(lstm_history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(lstm_history.history['loss'], label='Train Loss')
plt.plot(lstm_history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

What we see here is the model's flat line for accuracy, and the loss also remains relatively flat after the model stops learning.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Predict on validation set
y_pred_probs = lstm_model.predict(X_val)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# Step 2: Evaluation metrics
lstm_acc = accuracy_score(y_val, y_pred)
lstm_f1 = f1_score(y_val, y_pred)

print(f"Validation Accuracy: {lstm_acc:.4f}")
print(f"Validation F1 Score: {lstm_f1:.4f}")
print("\nClassification Report:\n", classification_report(y_val, y_pred))

# Step 3: Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
labels = ['Not Disaster', 'Disaster']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

From this, we see that the LSTM model has completely collapsed and is producing no meaningful data. It has performed much worse than the basic model. Let us move on to GRU.

## GRU Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

gru_callbacks_list = [
    EarlyStopping(
        monitor='val_loss', 
        patience=5, 
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=2, 
        min_lr=1e-6, 
        verbose=1
    ),
    ModelCheckpoint(
        filepath='gru_model.keras', 
        monitor='val_accuracy', 
        save_best_only=True, 
        verbose=1
    )
]

gru_history = gru_model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=gru_callbacks_list
)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(gru_history.history['accuracy'], label='Train Accuracy')
plt.plot(gru_history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(gru_history.history['loss'], label='Train Loss')
plt.plot(gru_history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

This model's history for accuracy and model loss looks remarkably similar to that of the LSTM. This model has also collapsed.

In [ ]:
y_pred_probs = gru_model.predict(X_val)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

gru_acc = accuracy_score(y_val, y_pred)
gru_f1 = f1_score(y_val, y_pred)
print(f"Validation Accuracy: {gru_acc:.4f}")
print(f"Validation F1 Score: {gru_f1:.4f}")
print(classification_report(y_val, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('GRU Confusion Matrix')
plt.show()

From this, we can see that the GRU suffers the same problem as the LSTM. So, we will go through and tweak some hyperparameters for the SimpleRNN.

## SimpleRNN Tuning - SpatialDropout1D

Let us add a SpatialDropout1D layer. A SpatialDropout1D is typically used with models such as GRU or LSTM, but we will try it with the SimpleRNN and see how it performs.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

BATCH_SIZE = 64

simple_model_v2 = Sequential([
    Embedding(input_dim=max_vocab, output_dim=128),
    SpatialDropout1D(0.2),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])
simple_model_v2.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

callbacks_list = [
    # This helps stop the model early
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    # Reduces learning rate when plateued
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    # save best model
    ModelCheckpoint(
        filepath='basic_model_v2.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

simple_history_v2 = simple_model_v2.fit(
    X_train, 
    y_train, 
    epochs=15, 
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=callbacks_list
)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(simple_history_v2.history['accuracy'], label='Train Accuracy')
plt.plot(simple_history_v2.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(simple_history_v2.history['loss'], label='Train Loss')
plt.plot(simple_history_v2.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

We can see that over time, the model has improved in accuracy, and the loss has decreased. 

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Predict on validation set
y_pred_probs = simple_model_v2.predict(X_val)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# Step 2: Evaluation metrics
basic_v2_acc = accuracy_score(y_val, y_pred)
basic_v2_f1 = f1_score(y_val, y_pred)

print(f"Validation Accuracy: {basic_v2_acc:.4f}")
print(f"Validation F1 Score: {basic_v2_f1:.4f}")
print("\nClassification Report:\n", classification_report(y_val, y_pred))

# Step 3: Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
labels = ['Not Disaster', 'Disaster']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

The F1 score has decreased.

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_val, y_pred)
basic_v2_roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {basic_v2_roc_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()

We can see that it decreased the F1 score from the original model. The AUC and the ROC are barely above the middle line, indicating that this model has also not performed well. Therefore, we will now remove this and replace it with a Dropout instead.

## SimpleRNN Tuning - Dropout

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

BATCH_SIZE = 64

simple_model_v3 = Sequential([
    Embedding(input_dim=max_vocab, output_dim=128),
    SimpleRNN(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])
simple_model_v3.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

callbacks_list = [
    # This helps stop the model early
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    # Reduces learning rate when plateued
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    # save best model
    ModelCheckpoint(
        filepath='basic_model_v3.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

simple_history_v3 = simple_model_v3.fit(
    X_train, 
    y_train, 
    epochs=15, 
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=callbacks_list
)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(simple_history_v3.history['accuracy'], label='Train Accuracy')
plt.plot(simple_history_v3.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(simple_history_v3.history['loss'], label='Train Loss')
plt.plot(simple_history_v3.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

In the plots above, we can see that training accuracy has increased significantly and loss has decreased, while validation accuracy has remained relatively unchanged. It could suggest overfitting to the data.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Predict on validation set
y_pred_probs = simple_model_v3.predict(X_val)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# Step 2: Evaluation metrics
basic_v3_acc = accuracy_score(y_val, y_pred)
basic_v3_f1 = f1_score(y_val, y_pred)

print(f"Validation Accuracy: {basic_v3_acc:.4f}")
print(f"Validation F1 Score: {basic_v3_f1:.4f}")
print("\nClassification Report:\n", classification_report(y_val, y_pred))

# Step 3: Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
labels = ['Not Disaster', 'Disaster']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

The F1 score has decreased compared to our original basic model, suggesting that it has not yet improved.

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_val, y_pred)
basic_v3_roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {basic_v3_roc_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()

We can also see from the ROC curve and AUC that it has got worse. Let us do one final tuning by adding regularization.

## SimpleRNN Tuning - L2 Regularization

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

BATCH_SIZE = 64

simple_model_v4 = Sequential([
    Embedding(input_dim=max_vocab, output_dim=128),
    SimpleRNN(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid', kernel_regularizer=l2(0.001))
])
simple_model_v4.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

callbacks_list = [
    # This helps stop the model early
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    # Reduces learning rate when plateued
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    # save best model
    ModelCheckpoint(
        filepath='basic_model_v4.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

simple_history_v4 = simple_model_v4.fit(
    X_train, 
    y_train, 
    epochs=15, 
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=callbacks_list
)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(simple_history_v4.history['accuracy'], label='Train Accuracy')
plt.plot(simple_history_v4.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(simple_history_v4.history['loss'], label='Train Loss')
plt.plot(simple_history_v4.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

Now, the validation accuracy and training accuracy have improved. However, the train loss and val loss do not follow the same pattern. This suggests there may still be some overfitting in the training data.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Predict on validation set
y_pred_probs = simple_model_v4.predict(X_val)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# Step 2: Evaluation metrics
basic_v4_acc = accuracy_score(y_val, y_pred)
basic_v4_f1 = f1_score(y_val, y_pred)

print(f"Validation Accuracy: {basic_v4_acc:.4f}")
print(f"Validation F1 Score: {basic_v4_f1:.4f}")
print("\nClassification Report:\n", classification_report(y_val, y_pred))

# Step 3: Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
labels = ['Not Disaster', 'Disaster']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

The F1 score has improved. We can also see that the model distinguishes between disasters and non-disasters more effectively than the previous models.

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_val, y_pred)
basic_v4_roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {basic_v4_roc_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()

Again, we can see that the ROC curve is looking much better, and the AUC is also better. It has considerable room for improvement, as the AUC is similar to that of the original basic model.

Let us check how this tuned model performs against the test data.

In [ ]:
test_sequences = tokenizer.texts_to_sequences(test['clean_text'])
X_test = pad_sequences(test_sequences, maxlen=max_len, padding='post', truncating='post')

y_test_probs = simple_model_v4.predict(X_test)
y_test_pred = (y_test_probs > 0.5).astype(int).flatten()

submission = pd.DataFrame({
    'id': test['id'],
    'target': y_test_pred
})

submission.to_csv('submission.csv', index=False)

The submission received a score of 0.71774. The score is quite good for a model that, on its own test/train dataset, ended up with a lower score than that.

# Conclusion and Learning

In [ ]:
results = {
    'Model': ['Basic Model', 'LSTM Model', 'GRU Model', 'Basic Model v2', 'Basic Model v3', 'Basic Model v4'],
    'Accuracy': [basic_acc, lstm_acc, gru_acc, basic_v2_acc, basic_v3_acc, basic_v4_acc],
    'F1': [basic_f1, lstm_f1, gru_f1, basic_v2_f1, basic_v3_f1, basic_v4_f1],
    'AUC': [basic_roc_auc, 'N/A', 'N/A', basic_v2_roc_auc, basic_v3_roc_auc, basic_v4_roc_auc]
}

df_result = pd.DataFrame(results).sort_values(by=['F1'], ascending=False)
df_result

We observed significant overfitting on the training data. The validation accuracy only increased slightly, while the training accuracy skyrocketed. This was also evident in the loss, where the validation loss remained relatively constant, but the training loss decreased significantly.

There are many ways we can still tune these models:

* Try different optimizers and even customize the starting learning rate.
* Add additional layers.
* Tune the input (text) by cleaning it further or even adding some of the relevant keywords or locations.

## Conclusion

The model received a decent score from the Kaggle competition, but there is still so much to be improved. It would be beneficial to investigate why the LSTM and GRU are collapsing and explore potential solutions to address this issue. That will mean more hyperparameter tuning - changing the optimizer for them. Overall, it is a good start for NLP.

One thing to note: Because the data set is so small, every time these models are run, they produce wildly different results. The only consistency is LSTM and GRU collapsing.

# References

Aitken, A. M. (2025). CNN Cancer Detection, 2. Retrieved 06/23/2025 from https://www.kaggle.com/code/alexandermaitken/cnn-cancer-detection#CNN-Cancer-Detection

Howard, A., devrishi, Culliton, P., & Guo, Y. (2019). Natural Language Processing with Disaster Tweets, 1. Retreieved 05/13/2025 from https://kaggle.com/competitions/nlp-getting-started

You can find a copy of the notebook in my Github: https://github.com/alexmaitken/csca5642-week4